In [1]:
# Transformer Architecture from Scratch
# A complete decoder-only Transformer implementation built from scratch using PyTorch.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

In [4]:
# 1. Multi-Head Self-Attention

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        assert self.head_dim * n_heads == d_model, "d_model must be divisible by n_heads"

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        # Linear projections & reshape for multi-head: (B, Seq_Len, N_Heads, Head_Dim) -> (B, N_Heads, Seq_Len, Head_Dim)
        q = self.q_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_linear(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, v)

        # Concatenate heads and put through final linear layer
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        return self.out(output)


In [5]:
# 2. Transformer Block & Feed-Forward Network

In [6]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.att = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # The Feed-Forward Network you asked for
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Attention sub-layer with residual connection & layer norm
        norm_x = self.norm1(x)
        x = x + self.dropout(self.att(norm_x, mask))

        # Feed-forward sub-layer with residual connection & layer norm
        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))
        return x

In [7]:
# 3. Mini-GPT Language Model Architecture

In [8]:
class MiniLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        batch_size, seq_len = idx.shape
        pos = torch.arange(0, seq_len, device=idx.device).unsqueeze(0)

        # Embeddings + Positional Encoding
        x = self.token_embedding(idx) + self.pos_embedding(pos)

        # Causal mask so tokens can't look into the future
        mask = torch.tril(torch.ones(seq_len, seq_len, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

In [9]:
# 4. Training Loop & Toy Dataset Execution

In [10]:
# Hyperparameters
vocab_size = 65
d_model = 64
n_heads = 4
n_layers = 2
max_len = 32
batch_size = 16

model = MiniLanguageModel(vocab_size, d_model, n_heads, n_layers, max_len)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Generate dummy text sequences for training
print("Initializing Training Loop...")
data = torch.randint(0, vocab_size, (1000, max_len + 1))

for epoch in range(5):
    total_loss = 0
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        if batch.shape[0] == 0: break

        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        optimizer.zero_grad()
        logits = model(inputs)

        # Reshape for cross entropy loss
        loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1} | Loss: {total_loss / (len(data) / batch_size):.4f}")

print("\nTraining complete! Your single-file Transformer is fully operational.")

Initializing Training Loop...
Epoch 1 | Loss: 4.2877
Epoch 2 | Loss: 4.2137
Epoch 3 | Loss: 4.2094
Epoch 4 | Loss: 4.2026
Epoch 5 | Loss: 4.1940

Training complete! Your single-file Transformer is fully operational.
